In [3]:
!pip install -q PyPDF2 aiofiles nest_asyncio

In [6]:
import os
import asyncio
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import logging
import nest_asyncio
from typing import List, Optional

# PyPDF2
from PyPDF2 import PdfReader

# optional async file writer
try:
    import aiofiles
    _HAVE_AIOFILES = True
except Exception:
    _HAVE_AIOFILES = False

# enable nested event loop for notebooks
nest_asyncio.apply()

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger("pypdf_extraction")

# --------- CONFIGURATION (edit these) ----------
PDF_SOURCE_DIR = "pdfs_20_2"        # relative to notebook cwd
OUTPUT_DIR = "pypdf_output_txt"      # relative to notebook cwd
MAX_WORKERS = 2                     # parallel worker threads (use 1 for low memory)
OFFSET = 0                          # skip first N files
LIMIT = 0                           # 0 = all
SEQUENTIAL = False                  # True => process one-by-one (lowest memory)
PAGE_MARKER = True                  # Insert page markers between pages
OVERWRITE = False                   # overwrite existing outputs
# ------------------------------------------------

def _safe_str(v: Optional[object]) -> str:
    return "" if v is None else str(v)

def extract_pdf_to_markdown(pdf_path: str, page_marker: bool = PAGE_MARKER) -> str:
    """
    Blocking extraction using PyPDF2.PdfReader.
    Returns a markdown-compatible string: metadata block + per-page text with optional markers.
    """
    try:
        reader = PdfReader(pdf_path)
    except Exception as e:
        logger.exception("Failed to open PDF %s: %s", pdf_path, e)
        return ""

    parts: List[str] = []

    # metadata
    try:
        meta = reader.metadata or {}
        if meta:
            parts.append("=== DOCUMENT METADATA ===")
            for k, v in meta.items():
                parts.append(f"{k}: {_safe_str(v)}")
            parts.append("")
    except Exception:
        pass

    # pages
    for i, page in enumerate(reader.pages, start=1):
        try:
            text = page.extract_text() or ""
        except Exception:
            text = ""
        if not text.strip():
            continue
        if page_marker:
            parts.append(f"\n---\n*Page {i}*\n---\n")
        parts.append(text.rstrip())
        parts.append("")

    return "\n".join(parts).strip()

def get_output_path(pdf_path: str, out_dir: str, ext: str = ".txt") -> str:
    base = Path(pdf_path).stem
    return os.path.join(out_dir, f"{base}{ext}")

def is_output_up_to_date(pdf_path: str, out_path: str) -> bool:
    if not os.path.exists(out_path):
        return False
    try:
        return os.path.getmtime(out_path) > os.path.getmtime(pdf_path)
    except OSError:
        return False

def _write_sync(path: str, content: str) -> None:
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        f.write(content)

async def _write_async(path: str, content: str, executor: Optional[ThreadPoolExecutor] = None) -> None:
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    if _HAVE_AIOFILES:
        async with aiofiles.open(path, "w", encoding="utf-8") as f:
            await f.write(content)
    else:
        loop = asyncio.get_event_loop()
        await loop.run_in_executor(executor, _write_sync, path, content)

async def _process_single(pdf_path: str, out_dir: str, executor: ThreadPoolExecutor, overwrite: bool) -> bool:
    out_path = get_output_path(pdf_path, out_dir, ext=".txt")
    if not overwrite and is_output_up_to_date(pdf_path, out_path):
        logger.info("Skipping (up-to-date): %s", os.path.basename(pdf_path))
        return True

    loop = asyncio.get_event_loop()
    md = await loop.run_in_executor(executor, extract_pdf_to_markdown, pdf_path, PAGE_MARKER)
    if not md:
        logger.warning("No text extracted: %s", os.path.basename(pdf_path))
        return False

    try:
        await _write_async(out_path, md, executor)
    except Exception:
        logger.debug("Async write failed, falling back to sync for %s", out_path)
        await loop.run_in_executor(executor, _write_sync, out_path, md)

    logger.info("Saved: %s", out_path)
    return True

async def process_folder_async(
    src_folder: str,
    out_folder: str,
    max_workers: int = MAX_WORKERS,
    offset: int = OFFSET,
    limit: int = LIMIT,
    sequential: bool = SEQUENTIAL,
    overwrite: bool = OVERWRITE,
):
    if not os.path.isdir(src_folder):
        raise FileNotFoundError(f"Source folder does not exist: {src_folder}")
    os.makedirs(out_folder, exist_ok=True)

    files = [f for f in os.listdir(src_folder) if f.lower().endswith(".pdf")]
    files.sort()
    if limit > 0:
        files = files[offset: offset + limit]
    else:
        files = files[offset:]

    if not files:
        logger.info("No PDFs to process in %s", src_folder)
        return

    logger.info("Processing %d PDFs -> %s (sequential=%s, workers=%d)", len(files), out_folder, sequential, max_workers)

    processed = 0
    failed: List[str] = []

    if sequential or max_workers <= 1:
        for fn in files:
            pdf_path = os.path.join(src_folder, fn)
            ok = await _process_single(pdf_path, out_folder, ThreadPoolExecutor(max_workers=1), overwrite)
            if ok:
                processed += 1
            else:
                failed.append(fn)
    else:
        executor = ThreadPoolExecutor(max_workers=max_workers)
        semaphore = asyncio.Semaphore(max_workers)

        async def worker(fn: str):
            async with semaphore:
                pdf_path = os.path.join(src_folder, fn)
                try:
                    ok = await _process_single(pdf_path, out_folder, executor, overwrite)
                    return (fn, ok, None)
                except Exception as e:
                    return (fn, False, str(e))

        tasks = [asyncio.create_task(worker(fn)) for fn in files]
        for fut in asyncio.as_completed(tasks):
            fn, ok, err = await fut
            if ok:
                processed += 1
                logger.info("✅ %s", fn)
            else:
                failed.append((fn, err))
                logger.error("❌ %s -> %s", fn, err)
        executor.shutdown(wait=True)

    logger.info("Done. Processed: %d  Failed: %d", processed, len(failed))
    if failed:
        logger.info("Failed sample: %s", failed[:10])

# ----------------- RUN (edit constants above) -----------------
cwd = os.getcwd()
src = os.path.join(cwd, PDF_SOURCE_DIR)
out = os.path.join(cwd, OUTPUT_DIR)

# In Jupyter: run the cell with `await` to start processing
await process_folder_async(src, out, max_workers=MAX_WORKERS, offset=OFFSET, limit=LIMIT, sequential=SEQUENTIAL, overwrite=OVERWRITE)

2025-10-19 08:29:44,552 INFO Processing 273 PDFs -> /home/huncho/Workspace/final_project_qna/chat_interface/pypdf_output_txt (sequential=False, workers=2)
2025-10-19 08:30:05,913 INFO Saved: /home/huncho/Workspace/final_project_qna/chat_interface/pypdf_output_txt/01_gcf-b42-02-add17-funding-proposal-package-fp275.txt
2025-10-19 08:30:06,074 INFO ✅ 01_gcf-b42-02-add17-funding-proposal-package-fp275.pdf
2025-10-19 08:30:13,985 INFO Saved: /home/huncho/Workspace/final_project_qna/chat_interface/pypdf_output_txt/02_gcf-b42-02-add16-funding-proposal-package-fp274.txt
2025-10-19 08:30:14,075 INFO ✅ 02_gcf-b42-02-add16-funding-proposal-package-fp274.pdf
2025-10-19 08:30:38,778 INFO Saved: /home/huncho/Workspace/final_project_qna/chat_interface/pypdf_output_txt/03_gcf-b42-02-add15-funding-proposal-package-fp273.txt
2025-10-19 08:30:38,842 INFO ✅ 03_gcf-b42-02-add15-funding-proposal-package-fp273.pdf
2025-10-19 08:30:48,757 INFO Saved: /home/huncho/Workspace/final_project_qna/chat_interface/pyp